# Browse saved workups

Use this notebook to find results by readable sample descriptions rather than folder names.
Each row represents one saved export. Metadata comes from that export's `manifest.json`,
not from today's configuration CSVs. Refreshing reads metadata only, skips Zarr chunk
directories, and does not change or delete any results.

The default view shows the newest saved export for each `scan_set_id`.
**Newest means latest save time, not best fit or scientifically preferred result.**
Switch `LATEST_ONLY` to `False` to see every saved version.

Discovery checks that a manifest and finalized consolidated Zarr metadata exist.
It does not repeat the workup notebook's full array verification.
Malformed or unfinished exports are listed separately and excluded from the main table.
Older `longC`/`longO` labels are displayed as `long` plus the energy region;
the original manifest is unchanged. Unrecorded energy regions remain unknown.


In [ ]:
from importlib import reload
import pandas as pd
import rsoxs_unified_workup.catalog as results_catalog

reload(results_catalog)

# None searches the project's results folder, including older layouts.
# You may instead supply an absolute path to another results folder.
RESULTS_ROOT = None

catalog = results_catalog.build_results_catalog(RESULTS_ROOT)

print("Results folder:", catalog.attrs["results_root"])
print(f"Saved exports: {len(catalog):,}")
print(f"Scan sets with saved results: {catalog['scan_set_id'].nunique():,}")

skipped = catalog.attrs["skipped_exports"]
if skipped:
    print("These locations were skipped; no files were changed:")
    display(pd.DataFrame(skipped))
elif catalog.empty:
    print("No saved exports found in this folder.")


## Find results

Leave a filter as `None` to include all values. Named filters match exactly,
ignoring capitalization. They may also be lists, such as `ENERGY_REGION = ["C", "O"]`.
`SEARCH` looks for a literal fragment of a sample description, sample ID, or scan-set ID.

For example, use `SCAN_TYPE = "long"` and `ENERGY_REGION = "O"` for long oxygen scans,
or `SAMPLE_ID = "ksr216"` for one sample.
Latest status is determined before these filters: an older export is not silently
substituted when a latest export fails a filter.

After a new workup is saved, rerun the refresh cell above and this cell.
The table displays at most `DISPLAY_LIMIT` rows; narrow the filters or increase that limit.


In [ ]:
LATEST_ONLY = True
SAMPLE_ID = None
SAMPLE_NAME = None
SCAN_TYPE = None
ENERGY_REGION = None
POLYMER = None
WASH_STATE = None
SEARCH = None
DISPLAY_LIMIT = 100

view = results_catalog.select_results(
    catalog,
    latest_only=LATEST_ONLY,
    sample_id=SAMPLE_ID,
    sample_name=SAMPLE_NAME,
    scan_type=SCAN_TYPE,
    energy_region=ENERGY_REGION,
    polymer=POLYMER,
    wash_state=WASH_STATE,
    search=SEARCH,
)

if isinstance(DISPLAY_LIMIT, bool) or not isinstance(DISPLAY_LIMIT, int) or DISPLAY_LIMIT < 1:
    raise ValueError("DISPLAY_LIMIT must be a positive integer")
print(f"Matching exports: {len(view):,}; showing {min(len(view), DISPLAY_LIMIT):,}")
print("View:", "latest save per scan set" if LATEST_ONLY else "all saved versions")

columns = [
    "sample_description", "scan_set_id", "saved_at_utc",
    "scan_ids", "group_count", "is_latest", "metadata_notes",
]
with pd.option_context("display.max_rows", DISPLAY_LIMIT, "display.max_colwidth", None):
    display(view.head(DISPLAY_LIMIT)[columns])


## Inspect one saved export

Set `EXPORT_ROW` to the numbered row at the left of the table.
This cell shows its exact saved location and exported groups, without loading arrays.
The row number is only a selector for the current filtered table; the paths and
`scan_set_id` are the identifiers to keep for later use.


In [ ]:
import json
from pathlib import Path

EXPORT_ROW = 0

if view.empty:
    raise ValueError("No exports match. Change the filters and rerun the table cell.")
if (
    isinstance(EXPORT_ROW, bool)
    or not isinstance(EXPORT_ROW, int)
    or not 0 <= EXPORT_ROW < len(view)
):
    raise ValueError(f"EXPORT_ROW must be between 0 and {len(view) - 1}")

selected_export = view.iloc[EXPORT_ROW].to_dict()
print("Sample:", selected_export["sample_description"])
print("Scan set:", selected_export["scan_set_id"])
print("Saved at (UTC):", selected_export["saved_at_utc"])
print("Output directory:", selected_export["export_directory"])
print("Zarr store:", selected_export["zarr_store"])
print("Manifest:", selected_export["manifest_path"])
print("Git commit at save:", selected_export["git_commit"])
print("Working tree clean at save:", selected_export["git_working_tree_clean"])
if selected_export["metadata_notes"]:
    print("Metadata notes:", selected_export["metadata_notes"])

with Path(selected_export["manifest_path"]).open(encoding="utf-8") as file:
    selected_manifest = json.load(file)

group_table = pd.DataFrame([
    {"group": group, "variables": ", ".join(variables)}
    for group, variables in selected_manifest["zarr_groups"].items()
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(group_table)


## Use a selection in later analysis

`selected_export["zarr_store"]` is the saved store path.
Choose an exact group name from the table above when opening it with xarray.
Each polarization pair retains its own native q grid; the catalog does not align
or combine data from different pairs or scan sets.

Keep `scan_set_id` and the selected export's path/save time if you want to return
to the same saved version later. A fresh catalog may mark a newer export as latest,
but it never replaces or deletes the older saved version.

No image arrays have been loaded by this notebook, and no exports have been modified.
